### This is to automatically add the cloze pattern to the cloze field

In [4]:
import requests
import json
import re

MULTI_LEMMA_TAG = "multi_lemma"
SINGLE_SUCCESS_TAG = "~single_lemma_generated"
SINGLE_FAILED_TAG = "~single_lemma_failed"

def invoke(action, **params):
    return requests.post(
        "http://localhost:8765",
        json={
            "action": action,
            "version": 6,
            "params": params
        }
    ).json()

# example: get deck names
print(invoke("deckNames"))


{'result': ['French MOVIES', 'Japanese Basic Vokab', 'Japanese Heisig', 'Japanese Heisig::All in one Kanji - Heisig order', 'Japanese Heisig::Deck in progress', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::Part 1: Stories (Lessons 1-12)', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::Part 2: Plots (Lessons 13-19)', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::Part 3: Elements (Lessons 20-56)', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::漢字 Outside 1st Volume', 'Japanese KANJI', 'Japanese KANJI::FirstLove', 'Japanese KANJI::LaLaLand', "Japanese KANJI::YUYU's ポッドキャスト", 'Japanese Media', 'Japanese Media::Filme & Serien', 'Japanese Media::Filme & Serien::FirstLove', 'Japanese Media::Filme & Serien::Heat', 'Japanese Media::Filme & Serien::LaLaLand', 'Japanese Media::Filme & Serien::ジブリ', 'Japanese Media::Filme & Serien::ジブリ::ハウルの動く城', 'Japanese Media::Filme & Serien::ジブリ:

In [5]:
#find all the Notes, that I am already learning in the "Japanese Media" deck, so that I can unsuspend their RTK card.
# take care of the fact that there might be more than one note type in this deck!

search_query = 'deck:"Japanese Media::Youtube::一つの動画::晩年の高畑勲と宮崎駿" is:new'
# If you want the whole parent deck instead, use: search_query = 'deck:"Japanese Media" is:review'

note_ids = invoke("findNotes", query=search_query)

#get information about those notes, to find the RTK card ids
notes_info = invoke("notesInfo", notes=note_ids['result'])
#DONT FORGET that in this deck are 2 different note types!!!

print(f"Matched notes: {len(note_ids['result'])}")
print(f"notesInfo returned: {len(notes_info['result'])}")

# field names by note type (important because this deck can contain multiple models)
fields_by_model = {}
for note in notes_info['result']:
    model = note.get('modelName', 'Unknown')
    fields_by_model.setdefault(model, set()).update(note['fields'].keys())

print("Detected fields by note type:")
for model, names in fields_by_model.items():
    print(model, sorted(names))






Matched notes: 56
notesInfo returned: 56
Detected fields by note type:
Moritz Language Reactor ['Audio Clip', 'Cloze', 'Color', 'Context', 'Context After Cloze', 'Context Before Cloze', 'Context Translation', 'Date Created', 'Date Modified', 'Diagram', 'Examples', 'Extra (unwichtig)', 'Extra2', 'Frequency', 'Furigana', 'Grammatik', 'Item ID', 'Item Key', 'Item Title', 'Kanji', 'Language', 'Lemma', 'Link', 'Next Image', 'Notes', 'Part of Speech', 'Phrase Transliteration', 'Pitch Accent', 'Prev Image', 'Question', 'Reading', 'Source', 'Subtitle', 'Subtitle Index', 'Translation', 'Translation Language', 'Word', 'Word Definition', 'Word Transliteration']


### Now we have the note ids and the field names, we can create the cloze deletion and update the notes with the new cloze field.

In [6]:



import re

def make_cloze(cloze_field, lemma, definition):
    # Skip rows where lemma contains a comma (handled by the multi-lemma block).
    if "," in lemma:
        return cloze_field, None

    new_cloze, replacements = re.subn(
        r"<strong>\s*(.*?)\s*</strong>",
        lambda m: f"{{{{c1::{m.group(1)}::{definition}}}}}",
        cloze_field,
        count=1,
    )
    return new_cloze, replacements > 0



for note in notes_info["result"]:
    note_id = note["noteId"]

    cloze_field = note["fields"]["Cloze"]["value"]  # adjust to your real field name
    Lemma = note["fields"]["Lemma"]["value"]
    Definition = note["fields"]["Word Definition"]["value"]  # adjust to your real field name

    new_sentence, generated = make_cloze(cloze_field, Lemma, Definition)

    update_result = invoke(
        "updateNoteFields",
        note={
            "id": note_id,
            "fields": {
                "Cloze": new_sentence
            }
        }
    )

    if generated is None:
        continue

    generated_successfully = generated and not update_result.get("error")
    if generated_successfully:
        invoke("addTags", notes=[note_id], tags=SINGLE_SUCCESS_TAG)
        invoke("removeTags", notes=[note_id], tags=SINGLE_FAILED_TAG)
    else:
        invoke("addTags", notes=[note_id], tags=SINGLE_FAILED_TAG)
        invoke("removeTags", notes=[note_id], tags=SINGLE_SUCCESS_TAG)


### Now we need to process the cards, that have several different words in the lemma field! 

1. add 2 new notes with an identical cloze field
2. then delete the 2 extra words from the lemma field
3. run the make_cloze function on these notes, (capture group needs to be slightly different)

TODO: automatically add the hiragana as hint

(japanese word instead of keyword?)

In [10]:
# MULTI-LEMMA PROCESSING (added by Codex)
# Splits notes where Lemma contains commas into multiple notes,
# keeps original Cloze context, and creates lemma-specific clozes.

import re

MULTI_LEMMA_TAG = "multi_lemma"
SINGLE_SUCCESS_TAG = "single_lemma_generated"
SINGLE_FAILED_TAG = "single_lemma_failed"

def split_lemmas(lemma_value):
    return [part.strip() for part in lemma_value.split(',') if part.strip()]


def merge_tags(tags, extra_tag):
    merged = set(tags or [])
    merged.add(extra_tag)
    return sorted(merged)


def make_cloze_for_lemma(cloze_field, lemma, definition):
    # Replace the <strong>...</strong> containing this lemma, but keep full captured text.
    pattern = re.compile(r"<strong>\s*(.*?)\s*</strong>")
    replaced = False

    def replace_matching_strong(match):
        nonlocal replaced
        strong_text = match.group(1)

        if not replaced and lemma and lemma in strong_text:
            replaced = True
            return f"{{{{c1::{strong_text}::{definition}}}}}"

        return match.group(0)

    updated = pattern.sub(replace_matching_strong, cloze_field)
    if replaced:
        return updated

    # Fallback if no strong tag contains the lemma.
    return pattern.sub(
        lambda m: f"{{{{c1::{m.group(1)}::{definition}}}}}",
        cloze_field,
        count=1,
    )


def get_note_fields(note):
    return {name: meta['value'] for name, meta in note['fields'].items()}


def get_deck_name_for_note(note):
    if note.get("deckName"):
        return note["deckName"]

    card_ids = note.get("cards", [])
    if not card_ids:
        return None

    cards_info = invoke("cardsInfo", cards=[card_ids[0]])
    if cards_info.get("error") or not cards_info.get("result"):
        return None

    return cards_info["result"][0].get("deckName")


for note in notes_info["result"]:
    note_id = note["noteId"]
    original_cloze = note["fields"]["Cloze"]["value"]
    lemma_value = note["fields"]["Lemma"]["value"]
    definition = note["fields"]["Word Definition"]["value"]

    lemmas = split_lemmas(lemma_value)

    if len(lemmas) <= 1:
        lemma = lemmas[0] if lemmas else lemma_value.strip()
        new_cloze = make_cloze_for_lemma(original_cloze, lemma, definition)
        invoke(
            "updateNoteFields",
            note={
                "id": note_id,
                "fields": {
                    "Lemma": lemma,
                    "Subtitle": original_cloze,
                    "Cloze": new_cloze,
                },
            },
        )
        continue

    # Keep first lemma on original note.
    first_lemma = lemmas[0]
    first_cloze = make_cloze_for_lemma(original_cloze, first_lemma, definition)
    invoke(
        "updateNoteFields",
        note={
            "id": note_id,
            "fields": {
                "Lemma": first_lemma,
                "Subtitle": original_cloze,
                "Cloze": first_cloze,
            },
        },
    )
    invoke("addTags", notes=[note_id], tags=MULTI_LEMMA_TAG)

    # Clone one new note per extra lemma.
    base_fields = get_note_fields(note)
    deck_name = get_deck_name_for_note(note)
    if not deck_name:
        print(f"Could not determine deck for note {note_id}; skipping clones")
        continue

    for extra_lemma in lemmas[1:]:
        cloned_fields = dict(base_fields)
        cloned_fields["Lemma"] = extra_lemma
        cloned_fields["Subtitle"] = original_cloze
        cloned_fields["Cloze"] = make_cloze_for_lemma(original_cloze, extra_lemma, definition)

        result = invoke(
            "addNote",
            note={
                "deckName": deck_name,
                "modelName": note["modelName"],
                "fields": cloned_fields,
                "tags": merge_tags(note.get("tags", []), MULTI_LEMMA_TAG),
            },
        )
        if result.get("error"):
            print(f"Failed to clone note {note_id} for lemma '{extra_lemma}': {result['error']}")
        else:
            print(f"Created note {result['result']} from note {note_id} for lemma '{extra_lemma}'")




In [11]:
# Add a tag to each note based on the last segment of its deck name.
# Example: Japanese Media::Filme & Serien::Isao Takahata on Grave of The Fireflies
# becomes tag: Isao_Takahata_on_Grave_of_The_Fireflies
def normalize_last_deck_segment_as_tag(deck_name):
    last_segment = deck_name.split("::")[-1].strip()
    return re.sub(r"\s+", "_", last_segment)


tagged = 0
skipped = 0

for note in notes_info["result"]:
    note_id = note["noteId"]
    deck_name = get_deck_name_for_note(note)

    if not deck_name:
        skipped += 1
        print(f"Skipped note {note_id}: no deck name found")
        continue

    tag_name = normalize_last_deck_segment_as_tag(deck_name)
    if not tag_name:
        skipped += 1
        print(f"Skipped note {note_id}: empty tag derived from deck '{deck_name}'")
        continue

    result = invoke("addTags", notes=[note_id], tags=tag_name)
    if result.get("error"):
        skipped += 1
        print(f"Failed to tag note {note_id} with '{tag_name}': {result['error']}")
        continue

    tagged += 1

print(f"Tagging complete. Tagged: {tagged}, skipped: {skipped}")


Tagging complete. Tagged: 56, skipped: 0


In [ ]:
# # Suspend all notes tagged SINGLE_FAILED_TAG and set their cards to green flag.
# FAILED_TAG = SINGLE_FAILED_TAG if "SINGLE_FAILED_TAG" in globals() else "single_lemma_failed"
# GREEN_FLAG = 3
# BATCH_SIZE = 500

# def chunked(seq, size):
#     for i in range(0, len(seq), size):
#         yield seq[i:i + size]


# failed_note_ids_resp = invoke("findNotes", query=f"tag:{FAILED_TAG}")
# failed_note_ids = failed_note_ids_resp.get("result", [])

# if failed_note_ids_resp.get("error"):
#     raise RuntimeError(f"findNotes failed: {failed_note_ids_resp['error']}")

# failed_card_ids = []
# if failed_note_ids:
#     notes_to_cards_resp = invoke("notesToCards", notes=failed_note_ids)
#     if notes_to_cards_resp.get("error"):
#         raise RuntimeError(f"notesToCards failed: {notes_to_cards_resp['error']}")
#     failed_card_ids = notes_to_cards_resp.get("result", [])

# for batch in chunked(failed_card_ids, BATCH_SIZE):
#     suspend_resp = invoke("suspend", cards=batch)
#     if suspend_resp.get("error"):
#         raise RuntimeError(f"suspend failed: {suspend_resp['error']}")

# for batch in chunked(failed_card_ids, BATCH_SIZE):
#     flag_resp = invoke("setFlag", cards=batch, flag=GREEN_FLAG)
#     if flag_resp.get("error"):
#         raise RuntimeError(f"setFlag failed: {flag_resp['error']}")

# print({
#     "failed_tag": FAILED_TAG,
#     "notes_found": len(failed_note_ids),
#     "cards_suspended": len(failed_card_ids),
#     "flag_set": "green",
# })
# -is

RuntimeError: notesToCards failed: unsupported action